In [1]:
import os
import gc
import torch
from pathlib import Path

from harreman_funcs import HarremanRunner
import harreman_summary

In [2]:
XENIUM_DATA_DIR = '/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium'
TIERS = ['Tier1', 'Tier2', 'Tier3']

In [3]:
def run_dataset(data_dir, dataset_name):
    harRunner = HarremanRunner(f'{data_dir}/{dataset_name}')
    harRunner.load_adata()
    harRunner.save_harreman_network()

    tiers = [tier for tier in TIERS if tier in harRunner.adata.obs.columns]
    if not tiers:
        raise ValueError('no tier annotations')

    for tier in tiers:
        harRunner.run_harreman(tier)

    out_path = harRunner.easy_download_path
    master, genepairs = harreman_summary.summarize_harreman_folder(out_path, sample_id=dataset_name)
    summary_dir = Path(out_path) / 'summary'
    summary_dir.mkdir(parents=True, exist_ok=True)
    master.to_csv(summary_dir / 'metabolite_summary.csv', index=False)
    genepairs.to_csv(summary_dir / 'gene_pair_summary.csv', index=False)
    harreman_summary.select_tcell_metabolites(out_path)

    # written last so a dataset is only skipped once it finished
    marker_path(data_dir, dataset_name).write_text(dataset_name)

In [4]:
def marker_path(data_dir, dataset_name):
    return Path(f'{data_dir}/{dataset_name}/easy_download/.{dataset_name}')


def run_all(data_dir=XENIUM_DATA_DIR):
    d_sets = [x for x in os.listdir(data_dir) if x not in ['FF_Human_Ovarian_Adenocarcinoma', 'FFPE_Human_Ovarian_Cancer']]
    # d_sets = ['Primary_Dermal_Melanoma']
    print(d_sets)
    for dataset_name in sorted(d_sets):
        if not os.path.isdir(f'{data_dir}/{dataset_name}'):
            continue
        if marker_path(data_dir, dataset_name).is_file():
            print(f'skipping {dataset_name}, already done')
            continue

        print(f'running {dataset_name}')
        try:
            run_dataset(data_dir, dataset_name)
            print(f'finished {dataset_name}')
        except Exception as e:
            print(f'failed {dataset_name}: {type(e).__name__}: {e}')

        gc.collect()
        torch.cuda.empty_cache()

In [5]:
run_all()

['Human_Breast', 'Human_Lung', 'Human_Prostate_Adenocarcinoma', 'Human_Cervical_Cancer', 'Primary_Dermal_Melanoma']
running Human_Breast
running cell independent
Extracting interaction database...
Finished extracting interaction database in 0.380 seconds
Applying gene filtering...
Finished applying gene filtering in 0.000 seconds
Computing the neighborhood graph...
Computing the weights...
Finished computing the KNN graph in 7.962 seconds
Computing gene pairs...
Finished computing gene pairs in 0.133 seconds
Gene pairs to test: 429
Starting cell-cell communication analysis...
Running the parametric test...
failed Human_Breast: OutOfMemoryError: CUDA out of memory. Tried to allocate 2.24 GiB. GPU 0 has a total capacity of 10.57 GiB of which 849.06 MiB is free. Including non-PyTorch memory, this process has 9.73 GiB memory in use. Of the allocated memory 9.54 GiB is allocated by PyTorch, and 21.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try

Permutation test:   0%|          | 0/1000 [00:00<?, ?it/s]


failed Human_Lung: OutOfMemoryError: CUDA out of memory. Tried to allocate 884.00 MiB. GPU 0 has a total capacity of 10.57 GiB of which 829.06 MiB is free. Including non-PyTorch memory, this process has 9.75 GiB memory in use. Of the allocated memory 8.90 GiB is allocated by PyTorch, and 692.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
running Human_Prostate_Adenocarcinoma
running cell independent
Extracting interaction database...
Finished extracting interaction database in 0.392 seconds
Applying gene filtering...
Finished applying gene filtering in 0.000 seconds
Computing the neighborhood graph...
Computing the weights...
Finished computing the KNN graph in 1.913 seconds
Computing gene pairs...
Finished comp

Permutation test:   0%|          | 0/1000 [00:00<?, ?it/s]


failed Human_Prostate_Adenocarcinoma: OutOfMemoryError: CUDA out of memory. Tried to allocate 614.00 MiB. GPU 0 has a total capacity of 10.57 GiB of which 603.06 MiB is free. Including non-PyTorch memory, this process has 9.97 GiB memory in use. Of the allocated memory 9.17 GiB is allocated by PyTorch, and 639.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
skipping Primary_Dermal_Melanoma, already done
